# Chat with any PDF

In [11]:
!pip cache purge --quiet

In [12]:
!pip install langchain==0.3.27 \
             langchain-community==0.3.27 \
             langchain-core==0.3.79 \
             langchain-text-splitters==0.3.11 \
             langchain-huggingface==0.3.1 \
             langchain-openai==0.3.33 \
             langchain-singlestore==1.0.0 \
             pypdf==6.1.1 \
             sentence-transformers==5.1.1 --quiet

In [15]:
import logging
import pandas as pd
import re

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_singlestore.vectorstores import SingleStoreVectorStore
from sentence_transformers import CrossEncoder
from singlestoredb.management import get_secret

In [16]:
# Only log errors for key libraries
logging.getLogger("httpx").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

In [17]:
LLM_MODEL = "gpt-4.1-mini"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER = "cross-encoder/ms-marco-MiniLM-L-6-v2"

os.environ["OPENAI_API_KEY"] = get_secret("OPENAI_API_KEY")

In [18]:
def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

In [19]:
file_url = "https://www.ipcc.ch/report/ar6/syr/downloads/report/IPCC_AR6_SYR_SPM.pdf"

loader = PyPDFLoader(file_url)

docs = loader.load()

docs = [
    Document(page_content = clean_text(d.page_content), metadata = d.metadata)
    for d in docs
]

In [20]:
print(f"You have {len(docs)} pages in your document.")
print(f"There are {sum(len(doc.page_content) for doc in docs)} characters in your document.")

You have 42 pages in your document.
There are 146794 characters in your document.


In [21]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100,
    separators = ["\n\n", ".", "!", "?", ";", ",", " "]
)

texts = text_splitter.split_documents(docs)

print (f"You have {len(texts)} pages.")

You have 181 pages.


In [22]:
llm = ChatOpenAI(
    model = LLM_MODEL,
    temperature = 0
)

In [23]:
embeddings = HuggingFaceEmbeddings(
    model_name = EMBEDDING_MODEL
)

dimensions = len(embeddings.embed_query("test"))

reranker = CrossEncoder(RERANKER)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [24]:
from sqlalchemy import *

db_connection = create_engine(connection_url)

In [26]:
with db_connection.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS pdf_docs;"))

In [27]:
vector_store = SingleStoreVectorStore(
    embeddings,
    table_name = "pdf_docs",
    distance_strategy = "DOT_PRODUCT",
    use_vector_index = True,
    vector_size = dimensions
)

vector_store.add_documents(texts);

In [28]:
df = pd.read_sql(
    """
    SELECT LEFT(content, 30) AS content, LEFT(vector :> JSON, 30) AS vector
    FROM pdf_docs
    LIMIT 5;
    """,
    db_connection
)

In [29]:
df.head()

,content,vector
0,"Summary for Policymakers IPCC,","[-0.0499601401,0.0395473056,0."
1,". In panel (b), the unit is th","[-0.0100800302,-0.0109694507,0"
2,". (high confidence) {3.3.3, 4.","[0.00129656564,0.0445485115,0."
3,. 5 °C Low emissions System tr,"[0.0708026811,0.0139502343,0.0"
4,18 Summary for Policymakers Su,"[0.00355993141,-0.0325920954,0"


In [30]:
prompt = PromptTemplate(
    input_variables = ["text", "question"],
    template = (
        "Answer the question based only on the following excerpts:\n"
        "Question: {question}\n"
        "Excerpts:\n{text}\n\n"
        "Provide a short, clear answer (2–3 sentences)."
    )
)

In [31]:
# Main helper function

def answer_question(question, vector_store, max_chunk_chars = 600, k = 2):
    """
    Fast and relevant OpenAI-powered version.
    """
    results = vector_store.similarity_search(question, k = min(k, len(texts)))

    if not results:
        return "No relevant information found."

    scored = [(r, reranker.predict([(question, r.page_content)])[0]) for r in results]
    results = [r for r, _ in sorted(scored, key = lambda x: x[1], reverse = True)]

    # Filter out junk or short text
    filtered = []
    for r in results:
        chunk = re.sub(r'\s+', ' ', r.page_content).strip()
        if len(chunk) < 50 or not re.search(r'[a-zA-Z]{5,}', chunk):
            continue
        filtered.append(chunk)

    if not filtered:
        return "No meaningful content found in retrieved sections."

    merged = "\n\n---\n\n".join(c[:max_chunk_chars] for c in filtered[:k])

    final_prompt = prompt.format(question = question, text = merged)
    try:
        response = llm.invoke(final_prompt)
        return response.content.strip()
    except Exception as e:
        print(f"OpenAI fallback: {e}")
        return filtered[0][:300]

In [32]:
print(answer_question(
    "What are the main observed impacts of climate change on ecosystems and humans?",
    vector_store
    )
)

The main observed impacts of climate change on ecosystems and humans include widespread and substantial losses and damages to terrestrial, freshwater, and ocean ecosystems, with high confidence in attribution to human-caused climate change. These impacts threaten biodiversity, livelihoods, health, and well-being, and are expected to intensify and become increasingly irreversible without urgent mitigation and adaptation.


In [33]:
print(answer_question(
    "What does the report say about limiting global warming to 1.5°C or 2°C?",
    vector_store
    )
)

The report states that limiting global warming to 1.5°C with no or limited overshoot is possible with immediate and deep global greenhouse gas emissions reductions this decade, but warming is still likely to exceed 1.5°C during the 21st century. Limiting warming to 2°C is more achievable (>67% likelihood) with immediate action, though current nationally determined contributions (NDCs) and policies make it harder to stay below these limits.


In [34]:
print(answer_question(
    "Which strategies are recommended to reduce greenhouse gas emissions?",
    vector_store
    )
)

Recommended strategies to reduce greenhouse gas emissions include implementing international environmental and sectoral agreements and initiatives that stimulate low GHG emissions investments. Achieving net zero CO2 or GHG emissions by mid-century is essential to limit global warming to 1.5°C or 2°C, requiring substantial emission reductions across sectors.


In [35]:
print(answer_question(
    "What adaptation measures are suggested for communities vulnerable to climate change?",
    vector_store
    )
)

Adaptation measures for vulnerable communities include integrating climate adaptation into social protection programs such as cash transfers and public works, and implementing equity-focused, inclusive, and rights-based approaches. Effective options also involve community-based adaptation, agricultural improvements like cultivar enhancements, water management, soil conservation, and landscape diversification.


In [36]:
print(answer_question(
    "What guidance does the report give to policymakers regarding climate action?",
    vector_store
    )
)

The report advises policymakers to urgently implement ambitious and immediate climate actions to limit global warming, emphasizing the integration of mitigation and adaptation strategies. It highlights the need for coordinated efforts across sectors and scales to reduce greenhouse gas emissions and enhance resilience to climate impacts.
